In [20]:
import os
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Load dataset
data_dir = "20_newsgroups"
categories = sorted(os.listdir(data_dir)) 
# print(categories)


In [21]:
# Preprocessing for the dataset
def strip_headers(text):
    """
    Remove headers from a raw newsgroup post.
    Headers end at the first blank line.
    """
    lines = text.split("\n")
    for i, line in enumerate(lines):
        if line.strip() == "":
            return "\n".join(lines[i+1:])
    return text

def strip_footers(text):
    """
    Remove footers (signatures separated by '--' or long trailing blocks).
    """
    return re.split(r"\n--+\n", text)[0]

def strip_quotes(text):
    """
    Remove quoted lines starting with '>'.
    """
    return "\n".join([line for line in text.split("\n") if not line.strip().startswith(">")])

def clean_text(raw_text):
    """
    Apply all cleaning steps: headers, footers, quotes.
    """
    text = strip_headers(raw_text)
    text = strip_footers(text)
    text = strip_quotes(text)
    return text.strip()

In [22]:
# Load the Raw Texts and Clean the text (remove unnessary part of the texts in news document)
# Handling missing values & Encoding Categorical Variables & Some Feature Engineering
news_texts = []
labels = []
for i, category in enumerate(categories): # encode the 20 newgroup category names to binary (0~19)
    category_path = os.path.join(data_dir, category)
    for file_name in os.listdir(category_path):
        filepath = os.path.join(category_path, file_name)
        try:
            with open(filepath, encoding="latin1") as f:
                texts = f.read()
                cleaned = clean_text(texts) # Feature Engineering
                if cleaned: # skip the empty news document (remove the data with missing values)
                    news_texts.append(cleaned)
                    labels.append(i)
        except Exception as e:
            print("Error reading:", filepath, e)

In [23]:
# Removing duplicate records
unique_texts = []
unique_labels = []
seen = set()

for texts, label in zip(news_texts, labels):
    if texts not in seen:
        unique_texts.append(texts)
        unique_labels.append(label)
        seen.add(texts)

news_texts, labels = unique_texts, unique_labels

In [24]:
# Detecting and Treating Outliers
filtered_news_texts = []
filtered_labels = []

for texts, label in zip(news_texts, labels):
    if len(texts.split()) > 10:
        filtered_news_texts.append(texts)
        filtered_labels.append(label)

news_texts, labels = filtered_news_texts, filtered_labels

In [25]:
# More Feature Engineering
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000
)

X = vectorizer.fit_transform(news_texts)

In [26]:
# Feature scaling or normalization
from sklearn.preprocessing import Normalizer
X = Normalizer().fit_transform(X)

In [ ]:
# Feature selection
from sklearn.feature_selection import chi2, SelectKBest

selector = SelectKBest(chi2, k=5000)
X = selector.fit_transform(X, labels)

In [ ]:
# Training and testing data split
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, stratify=labels, random_state=42
)

In [29]:
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Sample features:", vectorizer.get_feature_names_out()[:20])

Vocabulary size: 20000
Sample features: ['00' '000' '0001' '00010001b' '000iu' '001' '00100010b' '0062' '007' '01'
 '01000100b' '0114' '01730' '01752' '01wb' '02' '02106' '02139' '02238'
 '02p']
